# Model Load

In [1]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" "datasets>=3.0.0" "qwen-vl-utils"
!pip -q install "pandas==2.2.2" "pillow<12"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!unzip -q /content/2026_15_2_ai_DataSet.zip -d /content/

replace /content/dev.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A y


In [3]:
!unzip -q /content/qwen3_vl_32b_lora.zip -d /content/

replace /content/content/qwen3_vl_32b_lora/tokenizer_config.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: A y


In [4]:
!ls /content
!find /content -name "train.csv"
!find /content -name "adapter_model.safetensors"

2026_15_2_ai_DataSet.zip  qwen3_vl_32b_lora.zip  test.csv
content			  sample_data		 train
dev			  sample_submission.csv  train.csv
dev.csv			  test
/content/train.csv
/content/content/qwen3_vl_32b_lora/adapter_model.safetensors


In [5]:
import os
import random
import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftConfig, PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

CSV_PATH = "/content/train.csv"          # 필요시 경로 수정
IMAGE_ROOT = "/content"                  # 필요시 경로 수정
ADAPTER_PATH = "/content/content/qwen3_vl_32b_lora"

full_train_df = pd.read_csv(CSV_PATH)

# 중요: 원본 index 유지한 채 먼저 샘플링
sampled_train_df = full_train_df.sample(n=200, random_state=SEED)
remaining_df = full_train_df.drop(sampled_train_df.index)

train_df = sampled_train_df.reset_index(drop=True)
eval_df = remaining_df.sample(n=100, random_state=SEED).reset_index(drop=True)

print("DEVICE:", DEVICE)
print("full_train_df:", len(full_train_df))
print("sampled train_df:", len(train_df))
print("eval_df:", len(eval_df))
print(eval_df.head(2))

DEVICE: cuda
full_train_df: 5073
sampled train_df: 200
eval_df: 100
               id                  path  \
0  train_1389.jpg  train/train_1389.jpg   
1  train_1149.jpg  train/train_1149.jpg   

                                question    a     b    c    d answer  
0   사진에 보이는 몬스터 에너지 음료 캔의 재활용 분류는 무엇인가요?   종이  플라스틱   유리   금속      d  
1  사진에 보이는 재활용 가능한 플라스틱 병의 뚜껑 색깔은 무엇인가요?  노란색   빨간색  파란색  초록색      d  


In [6]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n"
        f"(b) {b}\n"
        f"(c) {c}\n"
        f"(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

def resolve_img_path(p):
    if os.path.isabs(p):
        return p
    return os.path.join(IMAGE_ROOT, p)

def extract_choice(text: str) -> str:
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"

    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last

    for line in reversed(lines):
        tokens = line.replace(".", " ").replace(")", " ").split()
        for tok in tokens:
            if tok in ["a", "b", "c", "d"]:
                return tok

    return "a"

# 이미지 경로 확인
for p in eval_df["path"].head(3):
    real_path = resolve_img_path(p)
    print(real_path, os.path.exists(real_path))

/content/train/train_1389.jpg True
/content/train/train_1149.jpg True
/content/train/train_4387.jpg True


In [7]:
print(f"[{ADAPTER_PATH}] 모델 로딩 중...")

peft_config = PeftConfig.from_pretrained(ADAPTER_PATH)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

base_model = AutoModelForImageTextToText.from_pretrained(
    peft_config.base_model_name_or_path,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

processor = AutoProcessor.from_pretrained(
    ADAPTER_PATH,
    trust_remote_code=True
)

model_device = next(model.parameters()).device

print("✅ 모델 로딩 완료")
print("Base model:", peft_config.base_model_name_or_path)
print("Model device:", model_device)

[/content/content/qwen3_vl_32b_lora] 모델 로딩 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1058 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

✅ 모델 로딩 완료
Base model: Qwen/Qwen3-VL-32B-Instruct
Model device: cuda:0


In [8]:
row = eval_df.iloc[0]

img_path = resolve_img_path(row["path"])
print("img_path:", img_path)

img = Image.open(img_path).convert("RGB")
user_text = build_mc_prompt(
    str(row["question"]),
    str(row["a"]),
    str(row["b"]),
    str(row["c"]),
    str(row["d"])
)

messages = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
    {"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": user_text}
    ]}
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    k: v.to(model_device) if isinstance(v, torch.Tensor) else v
    for k, v in inputs.items()
}

with torch.no_grad():
    out_ids = model.generate(
        **inputs,
        max_new_tokens=2,
        do_sample=False,
        eos_token_id=processor.tokenizer.eos_token_id
    )

generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(inputs["input_ids"], out_ids)
]

output_text = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

pred = extract_choice(output_text)

print("raw output:", repr(output_text))
print("pred:", pred)
print("truth:", row["answer"])

img_path: /content/train/train_1389.jpg


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


raw output: 'd'
pred: d
truth: d


In [9]:
correct = 0
total = 0
results = []

print("추론 시작...")

for i in tqdm(range(len(eval_df)), desc="Evaluating", unit="sample"):
    row = eval_df.iloc[i]

    img_path = resolve_img_path(row["path"])
    img = Image.open(img_path).convert("RGB")

    user_text = build_mc_prompt(
        str(row["question"]),
        str(row["a"]),
        str(row["b"]),
        str(row["c"]),
        str(row["d"])
    )

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": user_text}
        ]}
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model_device) if isinstance(v, torch.Tensor) else v
        for k, v in inputs.items()
    }

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id
        )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs["input_ids"], out_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    pred = extract_choice(output_text)
    truth = str(row["answer"]).strip().lower()

    ok = (pred == truth)
    if ok:
        correct += 1
    total += 1

    results.append({
        "id": row["id"] if "id" in eval_df.columns else i,
        "question": row["question"],
        "pred": pred,
        "truth": truth,
        "correct": "O" if ok else "X"
    })

accuracy = correct / total * 100
print(f"\n정확도: {accuracy:.2f}% ({correct}/{total})")

result_df = pd.DataFrame(results)
save_path = "/content/eval_result_qwen3_vl_32b_trainsplit.csv"
result_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("저장 완료:", save_path)
print(result_df.head())

추론 시작...


Evaluating:   0%|          | 0/100 [00:00<?, ?sample/s]


정확도: 48.00% (48/100)
저장 완료: /content/eval_result_qwen3_vl_32b_trainsplit.csv
               id                               question pred truth correct
0  train_1389.jpg   사진에 보이는 몬스터 에너지 음료 캔의 재활용 분류는 무엇인가요?    d     d       O
1  train_1149.jpg  사진에 보이는 재활용 가능한 플라스틱 병의 뚜껑 색깔은 무엇인가요?    a     d       X
2  train_4387.jpg             사진 속 재활용 가능한 컵의 재질은 무엇인가요?    b     c       X
3  train_2870.jpg    책상 위에 있는 바나나맛 우유 용기의 재활용 분류는 무엇인가요?    a     a       O
4  train_1392.jpg             사진에 보이는 재활용 가능한 용기는 무엇인가요?    a     c       X
